# WindBench — Benchmark Results

Thesis-facing figures and tables. Structure:

1. **Part 1 — Main Benchmark** (multi-farm, all 72 horizons)
2. **Part 2 — Data-Scaling Experiment** (Kelmarsh only)
3. **Appendix** — full per-horizon and per-farm tables

Rather than singling out cherry-picked horizons, results are aggregated over the full forecast window and broken down into three operationally meaningful bins:

- **Short** (1–12 h) — intraday balancing
- **Medium** (13–36 h) — day-ahead market
- **Long** (37–72 h) — multi-day planning

All point comparisons use **skill score vs persistence** (Part 1) or **MAE** (Part 2). Figures saved to `notebooks/figures/` as 300 dpi PDF, tables as CSV.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns

# ── Academic plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'serif',
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.titleweight':   'bold',
    'axes.labelsize':     10,
    'legend.fontsize':     9,
    'legend.framealpha':   0.9,
    'legend.edgecolor':   '#cccccc',
    'xtick.labelsize':     9,
    'ytick.labelsize':     9,
    'axes.linewidth':      0.8,
    'grid.linewidth':      0.5,
    'grid.alpha':          0.35,
    'lines.linewidth':     1.8,
    'lines.markersize':    6,
    'figure.dpi':         140,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
})

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

RESULTS_SEQ  = '../experiments/results/seq2seq.csv'
RESULTS_SCAL = '../experiments/results/scaling.csv'

print('Setup complete.')

In [ ]:
# ── Model metadata ─────────────────────────────────────────────────────────

MODEL_LABELS = {
    'seq2seq_arima':         'ARIMA',
    'seq2seq_random_forest': 'Random Forest',
    'seq2seq_xgboost':       'XGBoost',
    'seq2seq_lstm':          'LSTM',
    'seq2seq_transformer':   'Transformer',
    'seq2seq_nbeats':        'N-BEATS',
    'seq2seq_nhits':         'N-HiTS',
    'arima':                 'ARIMA',
    'sarimax':               'ARIMA',
    'ridge':                 'Ridge',
    'random_forest':         'Random Forest',
    'xgboost':               'XGBoost',
    'mlp':                   'MLP',
    'lstm':                  'LSTM',
    'transformer':           'Transformer',
    'nbeats':                'N-BEATS',
    'nhits':                 'N-HiTS',
    'tcn':                   'TCN',
    'nlinear':               'NLinear',
    'dlinear':               'DLinear',
    'persistence':           'Persistence',
}

# Excluded from line/scatter plots where ARIMA's scale compresses other models
LINE_EXCLUDE = {'ARIMA'}

MODEL_ORDER = [
    'Persistence', 'ARIMA',
    'Ridge',
    'Random Forest', 'XGBoost',
    'MLP', 'NLinear', 'DLinear',
    'LSTM', 'TCN', 'Transformer', 'N-BEATS', 'N-HiTS',
]

MODEL_COLORS = {
    'Persistence':   '#aaaaaa',
    'ARIMA':         '#b15928',
    'Ridge':         '#f4a460',
    'Random Forest': '#33a02c',
    'XGBoost':       '#6ab04c',
    'MLP':           '#cab2d6',
    'NLinear':       '#fdbf6f',
    'DLinear':       '#ff7f00',
    'LSTM':          '#1f78b4',
    'TCN':           '#a6cee3',
    'Transformer':   '#6a3d9a',
    'N-BEATS':       '#e31a1c',
    'N-HiTS':        '#fb9a99',
}

MODEL_MARKERS = {
    'Persistence': 'x',
    'ARIMA': 'D', 'Ridge': 's',
    'Random Forest': 's', 'XGBoost': 'P',
    'MLP': 'v', 'NLinear': '^', 'DLinear': '<',
    'LSTM': 'o', 'TCN': 'h', 'Transformer': '*',
    'N-BEATS': 'd', 'N-HiTS': '>',
}

MODEL_CATEGORY = {
    'Persistence': 'Baseline',
    'ARIMA': 'Statistical',
    'Ridge': 'Linear',
    'Random Forest': 'Tree', 'XGBoost': 'Tree',
    'MLP': 'Deep NN', 'NLinear': 'Deep NN', 'DLinear': 'Deep NN',
    'LSTM': 'Deep NN', 'TCN': 'Deep NN', 'Transformer': 'Deep NN',
    'N-BEATS': 'Deep NN', 'N-HiTS': 'Deep NN',
}

def c(label): return MODEL_COLORS.get(label, '#888888')
def m(label): return MODEL_MARKERS.get(label, 'o')

def sort_models(labels, exclude=None):
    excl = exclude if exclude is not None else set()
    ordered = [x for x in MODEL_ORDER if x in labels and x not in excl]
    rest    = [x for x in labels if x not in MODEL_ORDER and x not in excl]
    return ordered + rest

# ── Horizon bins (operational windows) ─────────────────────────────────────
HORIZON_BINS = [
    ('Short',  1, 12),   # intraday balancing
    ('Medium', 13, 36),  # day-ahead market
    ('Long',   37, 72),  # multi-day planning
]
BIN_NAMES = [b[0] for b in HORIZON_BINS]

def assign_bin(h):
    for name, lo, hi in HORIZON_BINS:
        if lo <= h <= hi:
            return name
    return None

# ── LaTeX export helper ────────────────────────────────────────────────────
def save_latex(obj, path, caption=None, label=None, **kwargs):
    """Save DataFrame as booktabs LaTeX. ± → $\\pm$, em-dash → --."""
    out = obj.copy()
    try:
        out = out.replace({'±': r' $\pm$ ', '—': '--'}, regex=True)
    except Exception:
        pass
    defaults = dict(
        buf=str(path),
        na_rep='--',
        float_format=lambda x: f'{x:.3f}',
        escape=False,
        bold_rows=False,
        position='ht',
    )
    if caption is not None:
        defaults['caption'] = caption
    if label is not None:
        defaults['label'] = label
    defaults.update(kwargs)
    # `position` requires caption to render a full table env in modern pandas
    if 'caption' not in defaults:
        defaults.pop('position', None)
    out.to_latex(**defaults)

print('Model metadata defined.')

In [ ]:
# ── Load and normalise benchmark results ──────────────────────────────────
df = pd.read_csv(RESULTS_SEQ)
df['model_label'] = df['model'].map(MODEL_LABELS).fillna(df['model'])
df = df.dropna(subset=['rmse'])
df['horizon_bin'] = df['horizon'].map(assign_bin)

avail_models  = sort_models(df['model_label'].unique().tolist())
all_horizons  = sorted(df['horizon'].unique())   # all 72 lead times
farms         = df['farm'].unique().tolist()

has_mase = 'mase' in df.columns

print(f'Farms       : {farms}')
print(f'Lead times  : {min(all_horizons)}–{max(all_horizons)} ({len(all_horizons)} total)')
print(f'Horizon bins: {", ".join(f"{n} ({lo}-{hi}h)" for n,lo,hi in HORIZON_BINS)}')
print(f'Models      : {avail_models}')
print(f'Metrics     : {[c for c in df.columns if c not in ["farm","horizon","model","model_label","error","horizon_bin"]]}')

---
# Part 1 — Main Benchmark

11 models × 12 farms × 72 lead times. One headline table + one figure + one per-farm heatmap, then probabilistic and compute tables.

### Table 1 — Headline Skill Score by Horizon Bin

Skill score (vs persistence), averaged within each operational bin and then across all farms. The **All** column averages over the full 1–72 h window. Values are `mean ± std` across farms; **bold** marks the best per column.

In [ ]:
# Average within (farm, model, bin), then take mean ± std across farms.
per_farm_bin = (
    df.groupby(['farm', 'model_label', 'horizon_bin'])['skill_score']
    .mean().reset_index()
)
bin_stats = (
    per_farm_bin.groupby(['model_label', 'horizon_bin'])['skill_score']
    .agg(['mean', 'std']).reset_index()
)

per_farm_all = (
    df.groupby(['farm', 'model_label'])['skill_score'].mean().reset_index()
)
all_stats = (
    per_farm_all.groupby('model_label')['skill_score']
    .agg(['mean', 'std']).reset_index()
)

rows = []
for lbl in [m for m in MODEL_ORDER if m in bin_stats['model_label'].unique()]:
    row = {'Model': lbl}
    for bname in BIN_NAMES:
        sub = bin_stats[(bin_stats['model_label']==lbl) & (bin_stats['horizon_bin']==bname)]
        if sub.empty:
            row[bname] = '—'
        else:
            mu, sd = sub['mean'].iloc[0], sub['std'].iloc[0]
            row[bname] = f'{mu:+.3f} ± {sd:.3f}'
    a = all_stats[all_stats['model_label']==lbl]
    if a.empty:
        row['All (1–72 h)'] = '—'
    else:
        row['All (1–72 h)'] = f'{a["mean"].iloc[0]:+.3f} ± {a["std"].iloc[0]:.3f}'
    rows.append(row)

head_tbl = pd.DataFrame(rows).set_index('Model')

def _highlight_best(s):
    vals = pd.to_numeric(s.str.extract(r'([+-]?\d+\.\d+)')[0], errors='coerce')
    is_best = vals == vals.max()
    return ['font-weight: bold' if v else '' for v in is_best]

styled = head_tbl.style.apply(_highlight_best, axis=0).set_caption(
    f'Skill score (vs persistence), mean ± std across {len(farms)} farms'
)
display(styled)
head_tbl.to_csv(FIGURES_DIR / 'tab1_headline_skill.csv')
save_latex(head_tbl, FIGURES_DIR / 'tab1_headline_skill.tex',
           caption=f'Headline skill score by horizon bin, mean $\\pm$ std across {len(farms)} farms.',
           label='tab:headline_skill')
print('Saved → figures/tab1_headline_skill.{csv,tex}')

### Figure 1 — Skill Score vs Forecast Horizon

Skill at every lead time from 1 to 72 h, averaged across all farms. Shaded band = ±1 std across farms for the top model.

In [ ]:
agg_curve = (
    df.groupby(['model_label', 'horizon'])['skill_score']
    .agg(['mean', 'std']).reset_index()
)
plot_models = sort_models(agg_curve['model_label'].unique(), exclude=LINE_EXCLUDE)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axhline(0.0, color='#444', linestyle='--', linewidth=0.9, zorder=0)
ax.text(72, 0.005, 'persistence parity', fontsize=8, color='#444', ha='right', va='bottom')

for lbl in plot_models:
    s = agg_curve[agg_curve['model_label']==lbl].sort_values('horizon')
    if s.empty: continue
    ax.plot(s['horizon'], s['mean'], color=c(lbl), marker=m(lbl),
            markersize=3.5, linewidth=1.4, label=lbl)

top_lbl = (
    agg_curve.groupby('model_label')['mean'].mean()
    .drop(labels=[l for l in LINE_EXCLUDE if l in agg_curve['model_label'].unique()], errors='ignore')
    .idxmax()
)
s_top = agg_curve[agg_curve['model_label']==top_lbl].sort_values('horizon')
ax.fill_between(s_top['horizon'], s_top['mean']-s_top['std'], s_top['mean']+s_top['std'],
                color=c(top_lbl), alpha=0.15, zorder=0, label=f'{top_lbl} ±1σ across farms')

ax.set_xlabel('Lead time (h)')
ax.set_ylabel('Skill Score (vs persistence)')
ax.set_title('Skill Score vs Forecast Horizon', fontweight='bold')
ax.set_xlim(0, 73)
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='lower right', ncol=2, fontsize=8.5, framealpha=0.9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig1_skill_vs_horizon.pdf', bbox_inches='tight')
plt.show()
print('Saved → figures/fig1_skill_vs_horizon.pdf')

### Figure 2 — Per-Farm Skill Heatmap

Skill score averaged across **all 72 lead times** for every (model, farm) pair. Red = beats persistence overall, blue = worse. Columns sorted left-to-right by farm difficulty (mean skill across all models).

In [ ]:
# Mean skill across all 72 horizons per (model, farm)
pivot = (
    df.pivot_table(index='model_label', columns='farm', values='skill_score', aggfunc='mean')
)
row_order = [m for m in MODEL_ORDER if m in pivot.index]
pivot = pivot.loc[row_order]
col_order = pivot.mean(axis=0).sort_values(ascending=False).index.tolist()
pivot = pivot[col_order]

fig, ax = plt.subplots(figsize=(max(8, 0.6*len(pivot.columns)+3), 0.45*len(pivot)+1.5))
vmax = max(abs(pivot.values.min()), abs(pivot.values.max()))
sns.heatmap(
    pivot, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
    annot=True, fmt='.2f', annot_kws={'fontsize': 8},
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Skill score', 'shrink': 0.8}, ax=ax,
)
ax.set_xlabel(''); ax.set_ylabel('')
ax.set_title('Skill Score per (Model, Farm) — averaged over all 72 lead times', fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig2_per_farm_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved → figures/fig2_per_farm_heatmap.pdf')

### Figure 2b — Per-Farm Skill vs Forecast Horizon

One panel per farm; each line a model. Shaded vertical bands mark the Short/Medium/Long horizon bins. Lets the reader spot farms where the model ranking diverges from the cross-farm mean shown in Figure 1.

In [ ]:
plot_models = sort_models(df['model_label'].unique(), exclude=LINE_EXCLUDE)
farms_sorted = sorted(farms)
n_farms = len(farms_sorted)
n_cols = 4
n_rows = int(np.ceil(n_farms / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2*n_cols, 2.8*n_rows), sharex=True)
axes = np.atleast_2d(axes).flatten()

for ax, farm in zip(axes, farms_sorted):
    sub = df[df['farm']==farm]
    for name, lo, hi in HORIZON_BINS:
        ax.axvspan(lo, hi, color='#eeeeee', alpha=0.4, zorder=0)
    ax.axhline(0, color='#444', linestyle='--', linewidth=0.6, zorder=1)
    for lbl in plot_models:
        s = sub[sub['model_label']==lbl].groupby('horizon')['skill_score'].mean()
        if s.empty: continue
        ax.plot(s.index, s.values, color=c(lbl), linewidth=1.1, label=lbl, zorder=3)
    ax.set_title(farm.replace('pilot_','').replace('_',' ').title(), fontsize=9.5, fontweight='bold')
    ax.set_xlim(0, 73)
    ax.grid(axis='y', alpha=0.3)

for ax in axes[n_farms:]:
    ax.axis('off')

for i, ax in enumerate(axes[:n_farms]):
    col = i % n_cols
    if col == 0:
        ax.set_ylabel('Skill')
    if i + n_cols >= n_farms:
        ax.set_xlabel('Lead time (h)')

handles = [Line2D([0],[0], color=c(lbl), linewidth=1.6, label=lbl) for lbl in plot_models]
fig.legend(handles=handles, loc='lower center', ncol=min(7, len(handles)),
           bbox_to_anchor=(0.5, -0.02), frameon=True, fontsize=9)
fig.suptitle('Per-Farm Skill Score vs Forecast Horizon — shaded bands = Short / Medium / Long bins',
             fontsize=11, fontweight='bold', y=1.00)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig2b_per_farm_skill_vs_horizon.pdf', bbox_inches='tight')
plt.show()
print('Saved → figures/fig2b_per_farm_skill_vs_horizon.pdf')

### Table 1b — Per-Farm Best Model by Horizon Bin

Cross-farm aggregates hide farm-specific winners. For each farm and each horizon bin (plus the full 1–72 h window), the cell shows the **best model** by mean skill score and its skill value.

In [ ]:
# Per (farm, model, bin) skill, then per (farm, model) skill (all-horizon)
fb = df.groupby(['farm', 'model_label', 'horizon_bin'])['skill_score'].mean().reset_index()
fa = df.groupby(['farm', 'model_label'])['skill_score'].mean().reset_index()
fa['horizon_bin'] = 'All'
combined = pd.concat([fb, fa], ignore_index=True)

# Best model per (farm, bin)
idx = combined.groupby(['farm', 'horizon_bin'])['skill_score'].idxmax()
best = combined.loc[idx].copy()
best['cell'] = best.apply(lambda r: f"{r['model_label']} ({r['skill_score']:+.3f})", axis=1)
tbl1b = best.pivot(index='farm', columns='horizon_bin', values='cell')
tbl1b = tbl1b[BIN_NAMES + ['All']]
tbl1b.index = [f.replace('pilot_','').replace('_',' ').title() for f in tbl1b.index]
tbl1b.index.name = 'Farm'

display(tbl1b.style.set_caption('Best model (skill score) per farm × horizon bin'))
tbl1b.to_csv(FIGURES_DIR / 'tab1b_per_farm_best_model.csv')
save_latex(tbl1b, FIGURES_DIR / 'tab1b_per_farm_best_model.tex',
           caption='Best model (skill score) per farm $\\times$ horizon bin.',
           label='tab:per_farm_best')
print('Saved → figures/tab1b_per_farm_best_model.{csv,tex}')

### Table 2 — Probabilistic Evaluation

Split conformal intervals at 90% nominal coverage. Averaged across all farms and 72 horizons.
- **PICP@90** — empirical coverage (target ≈ 0.90)
- **MPIW@90** — mean interval width, normalised by training target range
- **CRPS** — Continuous Ranked Probability Score (lower = better)

In [ ]:
prob_cols = [c for c in ['picp_90','mpiw_90','crps'] if c in df.columns]
if not prob_cols:
    print('No probabilistic columns in seq2seq.csv.')
else:
    prob_tbl = (
        df[df['model_label']!='ARIMA']
        .groupby('model_label')[prob_cols].mean()
    )
    row_order = [m for m in MODEL_ORDER if m in prob_tbl.index and m!='ARIMA']
    prob_tbl = prob_tbl.loc[row_order].round(3)
    prob_tbl.columns = [{
        'picp_90': 'PICP@90',
        'mpiw_90': 'MPIW@90 (norm.)',
        'crps':    'CRPS (kWh)',
    }[c] for c in prob_tbl.columns]
    display(prob_tbl)
    prob_tbl.to_csv(FIGURES_DIR / 'tab2_probabilistic.csv')
    save_latex(prob_tbl, FIGURES_DIR / 'tab2_probabilistic.tex',
               caption='Probabilistic evaluation: PICP@90 (dimensionless coverage), MPIW@90 (normalised by training target range, dimensionless), CRPS (kWh). Averaged across farms and 72 horizons.',
               label='tab:probabilistic')
    print('Saved → figures/tab2_probabilistic.{csv,tex}')

---
# Part 2 — Data-Scaling Experiment

Each model is retrained on increasing fractions of the Kelmarsh training set (10% → 100%) and evaluated on the same held-out test set across all 72 horizons. Reveals **sample efficiency**: how much data each model needs to reach its asymptotic performance.

In [ ]:
try:
    ds = pd.read_csv(RESULTS_SCAL)
    ds['model_label'] = ds['model'].map(MODEL_LABELS).fillna(ds['model'])
    ds = ds.dropna(subset=['mae'])
    scal_models    = sort_models(ds['model_label'].unique().tolist())
    scal_fracs     = sorted(ds['fraction'].unique())
    scal_horizons  = sorted(ds['horizon'].unique())
    scal_farms     = sorted(ds['farm'].unique())
    print(f'Scaling rows : {len(ds)}')
    print(f'Farms        : {scal_farms}')
    print(f'Models       : {scal_models}')
    print(f'Fractions    : {scal_fracs}')
    print(f'Horizons     : {min(scal_horizons)}–{max(scal_horizons)} ({len(scal_horizons)})')
except FileNotFoundError:
    ds = None
    print('scaling.csv not found.')

### Table 4 — Sample Efficiency Headline

For each model, MAE averaged over **all 72 horizons** at 10% / 50% / 100% of the Kelmarsh training set, plus the **efficiency ratio** MAE@10% / MAE@100% (1.0 = already saturated at 10% data; values >> 1 = data-hungry).

In [ ]:
if ds is None:
    pass
else:
    fracs_pick = [f for f in (0.1, 0.5, 1.0) if f in scal_fracs]
    # MAE averaged over all 72 horizons per (model, fraction)
    sub = ds[ds['fraction'].isin(fracs_pick)]
    wide = sub.pivot_table(index='model_label', columns='fraction', values='mae', aggfunc='mean')
    row_order = [m for m in MODEL_ORDER if m in wide.index]
    wide = wide.loc[row_order]
    if 1.0 in wide.columns and 0.1 in wide.columns:
        wide['Efficiency (10%/100%)'] = wide[0.1] / wide[1.0]
    rename = {f: f'MAE @ {int(f*100)}% (kWh)' for f in fracs_pick}
    wide = wide.rename(columns=rename)
    wide = wide.round(2)
    if 'Efficiency (10%/100%)' in wide.columns:
        wide['Efficiency (10%/100%)'] = wide['Efficiency (10%/100%)'].round(3)
    display(wide.style.set_caption('Sample efficiency on Kelmarsh — MAE (kWh) averaged over all 72 horizons'))
    wide.to_csv(FIGURES_DIR / 'tab4_sample_efficiency.csv')
    save_latex(wide, FIGURES_DIR / 'tab4_sample_efficiency.tex',
               caption='Sample efficiency on Kelmarsh: MAE (kWh) averaged over all 72 horizons, at training fractions of 10\\%, 50\\%, 100\\%, and the efficiency ratio MAE@10\\% / MAE@100\\%.',
               label='tab:sample_efficiency')
    print('Saved → figures/tab4_sample_efficiency.{csv,tex}')

### Figure 3 — MAE vs Training-Set Size, by Horizon Bin

One panel per horizon bin; MAE averaged within the bin. Tree-based models typically saturate quickly; sequential deep networks improve over a wider range of training fractions, with the gap widest in the long bin.

In [ ]:
if ds is None:
    pass
else:
    ds_b = ds.copy()
    ds_b['horizon_bin'] = ds_b['horizon'].map(assign_bin)
    bin_agg = (
        ds_b.groupby(['model_label', 'fraction', 'horizon_bin'])['mae']
        .mean().reset_index()
    )

    fig, axes = plt.subplots(1, len(BIN_NAMES), figsize=(5*len(BIN_NAMES), 4.5), sharey=False)
    if len(BIN_NAMES) == 1:
        axes = [axes]

    plot_models = sort_models(scal_models, exclude=LINE_EXCLUDE)

    for ax, bname in zip(axes, BIN_NAMES):
        lo, hi = next((l, h) for n, l, h in HORIZON_BINS if n == bname)
        sub = bin_agg[bin_agg['horizon_bin'] == bname]
        for lbl in plot_models:
            s = sub[sub['model_label']==lbl].sort_values('fraction')
            if s.empty: continue
            # kWh → MWh for readability
            ax.plot(s['fraction']*100, s['mae']/1000, color=c(lbl), marker=m(lbl),
                    markersize=4.5, linewidth=1.5, label=lbl)
        ax.set_title(f'{bname} ({lo}–{hi} h)', fontweight='bold')
        ax.set_xlabel('Training data fraction (%)')
        ax.set_ylabel('MAE (MWh)' if ax is axes[0] else '')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
        ax.grid(axis='y', alpha=0.3)

    handles = [Line2D([0],[0], color=c(lbl), marker=m(lbl), linewidth=1.5,
                       markersize=5, label=lbl) for lbl in plot_models]
    fig.legend(handles=handles, loc='lower center', ncol=min(6, len(handles)),
               bbox_to_anchor=(0.5, -0.10), frameon=True, fontsize=9)
    fig.suptitle('Data Scaling on Kelmarsh — MAE vs Training-Set Size by Horizon Bin',
                 fontsize=11, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'fig3_scaling_mae.pdf', bbox_inches='tight')
    plt.show()
    print('Saved → figures/fig3_scaling_mae.pdf')

### Figure 4 — Normalized Learning Curves

For each model, MAE at fraction *f* is divided by MAE at *f*=100%, so all curves converge to 1.0 at the right. The dashed line marks the asymptote. **Flat curves saturate fast** (data-efficient); **steeper curves** keep improving with more data (data-hungry). End-of-line annotations show the absolute MAE@100% as a sanity check on the level being normalized to.

In [ ]:
if ds is None:
    pass
else:
    # Mean MAE across all 72 horizons per (model, fraction)
    agg = ds.groupby(['model_label', 'fraction'])['mae'].mean().reset_index()
    mae100 = agg[agg['fraction'] == 1.0].set_index('model_label')['mae']
    agg['mae_norm'] = agg.apply(lambda r: r['mae'] / mae100[r['model_label']], axis=1)

    plot_models = sort_models(agg['model_label'].unique(), exclude=LINE_EXCLUDE)

    # Sort by sample-efficiency: largest normalized MAE@10% on top → most data-hungry
    eff_score = (
        agg[agg['fraction'] == 0.1].set_index('model_label')['mae_norm']
        .reindex(plot_models)
    )
    plot_models = eff_score.sort_values(ascending=False).index.tolist()

    fig, ax = plt.subplots(figsize=(10, 5.8))
    ax.axhline(1.0, color='#444', linestyle='--', linewidth=0.9, zorder=0)
    ax.text(101, 1.0, 'asymptote', fontsize=8, color='#444', va='center')

    for lbl in plot_models:
        s = agg[agg['model_label'] == lbl].sort_values('fraction')
        ax.plot(s['fraction']*100, s['mae_norm'],
                color=c(lbl), marker=m(lbl), markersize=5,
                linewidth=1.6, label=lbl)

    # Stagger end-of-line labels to avoid overlap
    end_pts = [(lbl, agg[(agg['model_label']==lbl) & (agg['fraction']==0.1)]['mae_norm'].iloc[0])
               for lbl in plot_models]
    end_pts.sort(key=lambda x: -x[1])  # top-down
    for lbl, y10 in end_pts:
        ax.annotate(
            f"  {lbl}  ({mae100[lbl]/1000:.1f} MWh)",
            xy=(10, y10), xytext=(-2, 0), textcoords='offset points',
            color=c(lbl), fontsize=8.5, va='center', ha='right',
            fontweight='bold',
        )

    ax.set_xlabel('Training data fraction (%)')
    ax.set_ylabel('MAE(f) / MAE(100%)')
    ax.set_title(
        'Sample Efficiency — Normalized Learning Curves on Kelmarsh\n'
        '(steeper curves = more sensitive to training-set size; MAE@100% shown in MWh)',
        fontweight='bold',
    )
    ax.set_xlim(-25, 110)
    ax.set_xticks([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'fig4_sample_efficiency.pdf', bbox_inches='tight')
    plt.show()
    print('Saved → figures/fig4_sample_efficiency.pdf')

---
# Appendix

Full per-horizon and per-farm tables for completeness.

## Appendix A — Full Benchmark Table by Horizon Bin

Per-model means across all farms, broken down by horizon bin and over the full 1–72 h window.

In [ ]:
app_metrics = [c for c in ['rmse','mae','skill_score'] if c in df.columns]

binned = (
    df.groupby(['model_label', 'horizon_bin'])[app_metrics]
    .mean()
    .unstack('horizon_bin')
)
overall = df.groupby('model_label')[app_metrics].mean()
overall.columns = pd.MultiIndex.from_product([overall.columns, ['All']])
wide = pd.concat([binned, overall], axis=1)

# Order metrics outermost, then bins in operational order
bin_order = BIN_NAMES + ['All']
wide = wide.reindex(columns=pd.MultiIndex.from_product([app_metrics, bin_order]))
wide = wide.round(3)

row_order = [m for m in MODEL_ORDER if m in wide.index]
wide = wide.loc[row_order]
display(wide)
wide.to_csv(FIGURES_DIR / 'appA_benchmark_full.csv')
save_latex(wide, FIGURES_DIR / 'appA_benchmark_full.tex',
           caption='Full benchmark: per-model RMSE (kWh), MAE (kWh), and skill score (dimensionless), averaged across all farms, by horizon bin and over the full 1--72\\,h window. Note: farm capacities differ by more than an order of magnitude, so absolute RMSE/MAE values are dominated by the largest farms; skill score is the cross-farm-comparable metric.',
           label='tab:app_benchmark_full')
print('Saved → figures/appA_benchmark_full.{csv,tex}')

## Appendix B — Per-Farm Skill (averaged over all 72 horizons)

In [ ]:
per_farm = (
    df.pivot_table(index='model_label', columns='farm', values='skill_score', aggfunc='mean')
    .round(3)
)
row_order = [m for m in MODEL_ORDER if m in per_farm.index]
per_farm = per_farm.loc[row_order]
display(per_farm)
per_farm.to_csv(FIGURES_DIR / 'appB_per_farm_skill.csv')
save_latex(per_farm, FIGURES_DIR / 'appB_per_farm_skill.tex',
           caption='Per-farm skill score, averaged over all 72 lead times.',
           label='tab:app_per_farm_skill')
print('Saved → figures/appB_per_farm_skill.{csv,tex}')

## Appendix C — Full Scaling Table (Kelmarsh)

MAE per (model, horizon bin, fraction). Bins as defined in the introduction.

In [ ]:
if ds is None:
    pass
else:
    ds_b = ds.copy()
    ds_b['horizon_bin'] = ds_b['horizon'].map(assign_bin)
    scal_wide = (
        ds_b.groupby(['model_label','horizon_bin','fraction'])['mae']
        .mean()
        .unstack('fraction')
        .round(2)
    )
    # Reorder bin level
    scal_wide = scal_wide.reindex(
        pd.MultiIndex.from_product(
            [[m for m in MODEL_ORDER if m in scal_wide.index.get_level_values(0).unique()],
             BIN_NAMES],
            names=['model_label', 'horizon_bin']
        )
    )
    display(scal_wide)
    scal_wide.to_csv(FIGURES_DIR / 'appC_scaling_full.csv')
    save_latex(scal_wide, FIGURES_DIR / 'appC_scaling_full.tex',
               caption='Full scaling table on Kelmarsh: MAE (kWh) per (model, horizon bin, training fraction).',
               label='tab:app_scaling_full')
    print('Saved → figures/appC_scaling_full.{csv,tex}')

## Appendix D — Per-Farm Skill vs Forecast Horizon (Full Set)

One full-page panel per farm showing skill score at every lead time 1–72 h. Individual PDFs in `figures/per_farm_skill/`, and a single merged PDF (`appD_per_farm_skill.pdf`) for inclusion as an appendix.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

PER_FARM_DIR = FIGURES_DIR / 'per_farm_skill'
PER_FARM_DIR.mkdir(exist_ok=True)

plot_models = sort_models(df['model_label'].unique(), exclude=LINE_EXCLUDE)
farms_sorted = sorted(farms)

merged_pdf = FIGURES_DIR / 'appD_per_farm_skill.pdf'
with PdfPages(merged_pdf) as pdf:
    for farm in farms_sorted:
        sub = df[df['farm'] == farm]
        fig, ax = plt.subplots(figsize=(8.5, 5.5))

        for name, lo, hi in HORIZON_BINS:
            ax.axvspan(lo, hi, color='#eeeeee', alpha=0.4, zorder=0)
        ax.axhline(0, color='#444', linestyle='--', linewidth=0.8, zorder=1)
        ax.text(72, 0.005, 'persistence parity', fontsize=8, color='#444',
                ha='right', va='bottom')

        for lbl in plot_models:
            s = sub[sub['model_label'] == lbl].groupby('horizon')['skill_score'].mean()
            if s.empty:
                continue
            ax.plot(s.index, s.values, color=c(lbl), marker=m(lbl),
                    markersize=3.5, linewidth=1.4, label=lbl)

        farm_title = farm.replace('pilot_', '').replace('_', ' ').title()
        ax.set_title(f'{farm_title} — Skill Score vs Forecast Horizon', fontweight='bold')
        ax.set_xlabel('Lead time (h)')
        ax.set_ylabel('Skill Score (vs persistence)')
        ax.set_xlim(0, 73)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(loc='lower right', ncol=2, fontsize=8.5, framealpha=0.9)

        fig.tight_layout()
        # Individual PDF
        single_path = PER_FARM_DIR / f'{farm}_skill_vs_horizon.pdf'
        fig.savefig(single_path, bbox_inches='tight')
        # Page in merged PDF
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {len(farms_sorted)} individual PDFs → {PER_FARM_DIR}/')
print(f'Merged → {merged_pdf}')

In [ ]:
print('All figures and tables saved to:', FIGURES_DIR.resolve())